In [1]:
# what is an agent
# An LLM agent runs tools in loop to achieve a goal

In [2]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os

In [30]:
load_dotenv(override=True)

True

In [31]:
sarvamai_api_key = os.getenv("SARVAM_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    # base_url="https://api.sarvam.ai/v1",
    api_key=openai_api_key,
)

In [32]:
def show(text):
    try:
        Console().print(text)
    except Exception as e:
        print(text)

In [33]:
# some lists
todos = []
completed = []

In [34]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [35]:
get_todo_report()

''

In [36]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [37]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [38]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [39]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [ ]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                "type": "array",
                "items": {"type": "string"},
                "title": "Descriptions",
            }
        },
        "required": ["descriptions"],
        "additionalProperties": False,
    },
}

In [ ]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        "properties": {
            "index": {
                "description": "The 1-based index of the todo to mark as complete",
                "title": "Index",
                "type": "integer",
            },
            "completion_notes": {
                "description": "Notes about how you completed the todo in rich console markup",
                "title": "Completion Notes",
                "type": "string",
            },
        },
        "required": ["index", "completion_notes"],
        "type": "object",
        "additionalProperties": False,
    },
}

In [ ]:
tools = [
    {"type": "function", "function": create_todos_json},
    {"type": "function", "function": mark_complete_json},
]

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append(
            {
                "role": "tool",
                "content": json.dumps(result),
                "tool_call_id": tool_call.id,
            }
        )
    return results

In [51]:
def loop(messages):
    done = False
    while not done:
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b:free",
            messages=messages,
            tools=tools,
        )
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [52]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message},
]

In [53]:
todos, completed = [], []
loop(messages)

Todo #1: Assume distance between Boston and New York is approximately 215 miles (reasonable estimate).
Todo #2: Calculate distance traveled by Boston train before New York train departs.
Todo #3: Compute remaining distance at 3:00 pm.
Todo #4: Determine combined speed of both trains.
Todo #5: Calculate time after 3:00 pm until they meet.
Todo #6: Add the time to 3:00 pm to get meeting time.

Assumed distance between Boston and New York = 215 miles (typical rail distance).

Todo #1: Assume distance between Boston and New York is approximately 215 miles (reasonable estimate).
Todo #2: Calculate distance traveled by Boston train before New York train departs.
Todo #3: Compute remaining distance at 3:00 pm.
Todo #4: Determine combined speed of both trains.
Todo #5: Calculate time after 3:00 pm until they meet.
Todo #6: Add the time to 3:00 pm to get meeting time.

Boston train travels 60 miles in the first hour (60 mph × 1 hr).

Todo #1: Assume distance between Boston and New York is approximately 215 miles (reasonable estimate).
Todo #2: Calculate distance traveled by Boston train before New York train departs.
Todo #3: Compute remaining distance at 3:00 pm.
Todo #4: Determine combined speed of both trains.
Todo #5: Calculate time after 3:00 pm until they meet.
Todo #6: Add the time to 3:00 pm to get meeting time.

Remaining distance at 3:00 pm = 215 mi - 60 mi = 155 miles.

Todo #1: Assume distance between Boston and New York is approximately 215 miles (reasonable estimate).
Todo #2: Calculate distance traveled by Boston train before New York train departs.
Todo #3: Compute remaining distance at 3:00 pm.
Todo #4: Determine combined speed of both trains.
Todo #5: Calculate time after 3:00 pm until they meet.
Todo #6: Add the time to 3:00 pm to get meeting time.

Combined speed = 60 mph + 80 mph = 140 mph.

Todo #1: Assume distance between Boston and New York is approximately 215 miles (reasonable estimate).
Todo #2: Calculate distance traveled by Boston train before New York train departs.
Todo #3: Compute remaining distance at 3:00 pm.
Todo #4: Determine combined speed of both trains.
Todo #5: Calculate time after 3:00 pm until they meet.
Todo #6: Add the time to 3:00 pm to get meeting time.

RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'openai/gpt-oss-120b:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'OpenInference', 'is_byok': False}}, 'user_id': 'user_30gGEhSFVUDyqKFApFpS2pwtYbH'}

In [ ]:
# This is working fine and properly
# TODO:
# Try to build an agent loop from scratch
# recommended typing from scratch